<a href="https://colab.research.google.com/github/jess22jess/Ing_Fin/blob/main/C%C3%B3digo_Data_Cruda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Librerías

Aquí se instalan las librerías externas que el programa necesita para funcionar, ya que Python no las trae por defecto:

* **requests:** Para conectarse a internet y pedir los datos de la página.

* **beautifulsoup4 y lxml:** Para leer y entender el código HTML de la página web.

* **pandas:** Para armar la tabla de datos y limpiarla.

* **openpyxl:** Para guardar esa tabla como un archivo de Excel.

* **selenium y webdriver-manager:** Para controlar un navegador automáticamente.

In [ ]:
!pip install requests beautifulsoup4 pandas openpyxl lxml selenium webdriver-manager

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 105.1 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [selenium]


Importamos las herramientas que se instalaron en la celda anterior, para poder usarlas en el código:

* **requests, json, time, re:** Herramientas de Python para conectarse a la web, manejar texto y hacer pausas.

* **pandas:** Para crear y manipular la tabla de datos (el Excel).

* **BeautifulSoup:** Para leer el HTML de la página.

* **datetime:** Para poner la fecha y hora de extracción.

* **urljoin:** Para armar las URLs completas de los productos.

* **display:** Para mostrar la tabla más estilizada en la pantalla.

In [ ]:
import requests
import pandas as pd
import json
import re
import time

from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin
from IPython.display import display

# Configuración General

En esta celda el código:

**Define las URLs:** Guarda las direcciones de la página principal y de la API (espor donde saca los datos).

**Pone los límites:** Le dice al programa que se detenga al llegar a 5,500 productos, que lea de 24 en 24 por página (la paginación) y que haga una pausa de 0.7 segundos entre cada petición (para no ser detectados).

**Pone la fecha:** Genera la fecha y hora exacta de la extracción.

**Configura el "disfraz":** Crea los encabezados con el User-Agent de un navegador real (Chrome) para engañar a la página y así que parezca que es un humano. Esto es la solución al anti-websraping.

**Crea los contenedores:** Prepara diccionarios y listas vacías (categorias, productos_unicos, errores) que se irán llenando con la información conforme avance el scraper.

In [ ]:
URL_PRINCIPAL = "https://www.fahorro.com/farmacia.html"

URL_API = "https://api.empathy.co/search/v1/query/fda/browse"

URL_BASE = "https://www.fahorro.com/"

MAX_PRODUCTOS = 5500
PRODUCTOS_POR_PAGINA = 24
MAX_PAGINAS = 500
ESPERA = 0.7

FECHA_EXTRACCION = datetime.now().strftime(
    "%Y-%m-%d %H:%M:%S"
)

NOMBRE_ARCHIVO = "Farmacias_del_Ahorro_5500.xlsx"

encabezados = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "es-MX,es;q=0.9"
}

sesion = requests.Session()
sesion.headers.update(encabezados)

# Contenedores de información
categorias = {}
productos_unicos = {}
resumen_categorias = []
errores = []

# Funciones Auxiliares

**obtener_categoria_id:** Usa "expresiones regulares" (búsquedas avanzadas de texto) para escarbar en el HTML de la página y encontrar el número de ID de cada categoría. Sin ese ID, la API no devuelve los productos.


In [ ]:
def obtener_categoria_id(html):

    patrones = [
        r'"categoryId"\s*:\s*"?(\d+)"?',
        r'"category_id"\s*:\s*"?(\d+)"?',
        r'categoryId\s*[:=]\s*["\']?(\d+)'
    ]

    for patron in patrones:

        resultado = re.search(
            patron,
            html,
            re.IGNORECASE
        )

        if resultado:
            return resultado.group(1)

    return None

**obtener_url_producto:** Toma el nombre corto del producto y lo convierte en un link completo y válido. Esta función es clave porque es la que genera la *columna del URL.*


In [ ]:
def obtener_url_producto(slug):

    if not slug:
        return None

    slug = str(slug).strip()

    if slug.startswith("http"):
        return slug

    if slug.endswith(".html"):
        return urljoin(URL_BASE, slug)

    return urljoin(URL_BASE, slug + ".html")

**agregar_error:** Si algo falla, guarda el error en una lista con la etapa, la URL y el motivo, para que puedan revisarlo después en la hoja de "Errores" del Excel.

In [ ]:
def agregar_error(etapa, categoria, url, mensaje):

    errores.append({
        "Etapa_del_proceso": etapa,
        "Categoria": categoria,
        "URL_consultada": url,
        "Descripcion_del_error": str(mensaje)
    })

# Descubrimiento de Categorías

Este bloque es la Etapa 1:

Aquí el programa todavía no busca productos, sino que primero va a la página principal de Farmacias del Ahorro y descubre cuáles son todas las **categorías** que existen (Gastroenterología, Ortopedia, Dolor, etc.).

---

En pocas palabras:

Entra a la página principal y usa BeautifulSoup para leer todo el HTML.

Busca todos los enlaces $(<a>)$ que contengan **"/farmacia/"** y que terminen en **".html"**. Esos son los enlaces a las categorías.

Entra a cada categoría, busca su ID (usando la función auxiliar de antes) y guarda su nombre y su URL en el diccionario categorias.

Hace una pausa **(time.sleep)** entre cada categoría para no ser detectado.

Si alguna categoría falla, la guarda en el registro de errores.

Al final, se asegura de incluir manualmente la categoría "Dolor y Fiebre" (ID 9651) por si acaso no la encontró en el camino.

In [ ]:
print("=" * 65)
print("ETAPA 1: CATEGORIAS")
print("=" * 65)

try:

    respuesta = sesion.get(
        URL_PRINCIPAL,
        timeout=30
    )

    print("Codigo HTTP:", respuesta.status_code)

    respuesta.raise_for_status()

    sopa = BeautifulSoup(
        respuesta.text,
        "html.parser"
    )

    enlaces = sopa.find_all(
        "a",
        href=True
    )

    urls_candidatas = set()

    for enlace in enlaces:

        href = enlace["href"].strip()

        if (
            "/farmacia/" in href
            and href.endswith(".html")
        ):

            urls_candidatas.add(
                urljoin(URL_BASE, href)
            )

    urls_candidatas.add(URL_PRINCIPAL)

    print(
        "Paginas candidatas:",
        len(urls_candidatas)
    )

    for numero, url_categoria in enumerate(
        sorted(urls_candidatas),
        start=1
    ):

        try:

            pagina = sesion.get(
                url_categoria,
                timeout=30
            )

            if pagina.status_code != 200:

                agregar_error(
                    "Descubrimiento",
                    "",
                    url_categoria,
                    f"HTTP {pagina.status_code}"
                )

                continue

            categoria_id = obtener_categoria_id(
                pagina.text
            )

            if categoria_id:

                titulo = BeautifulSoup(
                    pagina.text,
                    "html.parser"
                ).title

                nombre_categoria = (
                    titulo.get_text(strip=True)
                    if titulo
                    else url_categoria.split("/")[-1]
                )

                categorias[categoria_id] = {
                    "categoria_id": categoria_id,
                    "categoria": nombre_categoria,
                    "url_categoria": url_categoria
                }

                print(
                    f"Categoria {len(categorias)}: "
                    f"{nombre_categoria} | ID: {categoria_id}"
                )

            time.sleep(ESPERA)

        except Exception as e:

            agregar_error(
                "Descubrimiento",
                "",
                url_categoria,
                e
            )


except Exception as e:

    agregar_error(
        "Pagina principal",
        "",
        URL_PRINCIPAL,
        e
    )


# Incorporar la categoria que ya comprobamos.
if "9651" not in categorias:

    categorias["9651"] = {
        "categoria_id": "9651",
        "categoria": "Dolor y Fiebre",
        "url_categoria": (
            "https://www.fahorro.com/farmacia/dolor-y-fiebre.html"
        )
    }

print("\nCategorias identificadas:", len(categorias))

# Parámetros de la API

En este apartado se definen los parámetros fijos que van en cada consulta:

**"origin": "search_box":** Indica que la petición simula venir de la caja de búsqueda de la página principal.

**"rows": PRODUCTOS_POR_PAGINA:** Define el número de productos que se devolverán por cada página de resultados.

**"lang": "es":** Especifica el idioma de los resultados, en este caso español.

**"scope": "mobile":** Solicita la versión móvil del catálogo, que suele ser más ligera y fácil de procesar.

**"channel": "tl":** Identifica el canal de la tienda en línea (Tienda en Línea).

**"region": "NACIONAL":** Limita la búsqueda a productos disponibles a nivel nacional, sin filtrar por sucursal.

**"customerId": "":** Se deja vacío para realizar la consulta como un usuario anónimo.

**"browseField": "categoryIds":** Indica que la búsqueda se filtrará utilizando el ID de la categoría obtenido en la etapa anterior.


In [ ]:
parametros_base = {
    "origin": "search_box",
    "rows": PRODUCTOS_POR_PAGINA,
    "lang": "es",
    "scope": "mobile",
    "channel": "tl",
    "region": "NACIONAL",
    "customerId": "",
    "browseField": "categoryIds"
}

# Extracción de Productos

Etapa 2. Aquí se extraen los productos (es un bloque largo porque tiene que ser muy cuidadoso para no perderse ni duplicar datos).

---

* Recorre cada categoría: El primer bucle **for categoria in categorias.values()** va una por una por todas las categorías que descubrieron en la Etapa 1 y se detiene al llegar al límite de 5,500 productos.

* Paginación (el segundo bucle): Para cada categoría, el programa simula que avanza página por página usando el parámetro *"start"* (0, 24, 48...). Aquí es donde se cumple el rubro de "Tabla paginada".

* Consulta a la API: Hace la petición a la API de Empathy usando la sesión que tiene el User-Agent de navegador real (evadiendo el anti-websraping).

* Evita páginas repetidas: Crea una "firma" con los SKUs de la página. Si esa firma ya la vio antes, significa que la página se repitió y se detiene para no hacer un bucle infinito.

* Extrae y limpia los datos: Para cada producto, revisa si tiene **SKU**, si no, usa su **ID**. Además:

  * Le agrega la columna **URL_producto**.

  * Le agrega la columna **fecha_extraccion**.

  * Le agrega la columna **pagina_origen** (para saber de qué página obtuvimos la información).

* Evita duplicados: Usa el diccionario **productos_unicos**. Si un producto ya existe, no lo vuelve a guardar; pero si lo encuentra en otra categoría, le agrega esa nueva categoría a la columna **categorias_encontradas**.

* Hace una pausa: Usa **time.sleep** (ESPERA) después de cada página para no saturar el servidor.

* Registra el resumen: Al terminar cada categoría, guarda cuántos resultados recuperó, cuántas páginas consultó y cuántos productos únicos lleva acumulados.

In [ ]:
print("\n" + "=" * 65)
print("ETAPA 2: EXTRACCION DE PRODUCTOS")
print("=" * 65)

for numero, categoria in enumerate(
    categorias.values(),
    start=1
):

    if len(productos_unicos) >= MAX_PRODUCTOS:

        print("\nSe alcanzo el limite de 5,500 productos.")
        break

    categoria_id = categoria["categoria_id"]
    nombre_categoria = categoria["categoria"]

    print(
        f"\n[{numero}/{len(categorias)}] "
        f"{nombre_categoria}"
    )

    total_resultados = 0
    paginas_consultadas = 0
    paginas_vistas = set()

    for inicio in range(
        0,
        MAX_PAGINAS * PRODUCTOS_POR_PAGINA,
        PRODUCTOS_POR_PAGINA
    ):

        if len(productos_unicos) >= MAX_PRODUCTOS:
            break

        parametros = parametros_base.copy()

        parametros["start"] = inicio
        parametros["browseValue"] = categoria_id

        try:

            respuesta = sesion.get(
                URL_API,
                params=parametros,
                timeout=40
            )

            if respuesta.status_code != 200:

                agregar_error(
                    "Consulta API",
                    nombre_categoria,
                    respuesta.url,
                    f"HTTP {respuesta.status_code}"
                )

                print(
                    "Solicitud detenida. HTTP:",
                    respuesta.status_code
                )

                break

            datos = respuesta.json()

            contenido = (
                datos.get("catalog", {})
                .get("content", [])
            )

            if not contenido:

                print("No hay mas productos en esta categoria.")
                break

            # Evitar recorrer una pagina repetida.
            firma_pagina = tuple(
                str(
                    producto.get("sku")
                    or producto.get("ecommId")
                    or producto.get("ecommUrlKey")
                )
                for producto in contenido
            )

            if firma_pagina in paginas_vistas:

                print("Pagina repetida. Se detiene la categoria.")
                break

            paginas_vistas.add(firma_pagina)

            paginas_consultadas += 1

            nuevos_en_pagina = 0

            for producto in contenido:

                if len(productos_unicos) >= MAX_PRODUCTOS:
                    break

                sku = producto.get("sku")

                identificador = (
                    str(sku)
                    if sku
                    else str(
                        producto.get("ecommId")
                        or producto.get("ecommUrlKey")
                    )
                )

                if not identificador or identificador == "None":
                    continue

                if identificador not in productos_unicos:

                    registro = producto.copy()

                    registro["URL_producto"] = (
                        obtener_url_producto(
                            producto.get("ecommUrlKey")
                        )
                    )

                    registro["URL_categoria"] = (
                        categoria["url_categoria"]
                    )

                    registro["categoria_origen"] = (
                        nombre_categoria
                    )

                    registro["category_id_origen"] = (
                        categoria_id
                    )

                    registro["categorias_encontradas"] = (
                        nombre_categoria
                    )

                    registro["fecha_extraccion"] = (
                        FECHA_EXTRACCION
                    )

                    registro["pagina_origen"] = (
                        inicio // PRODUCTOS_POR_PAGINA + 1
                    )

                    productos_unicos[identificador] = registro

                    nuevos_en_pagina += 1

                else:

                    # Registrar las categorias adicionales.
                    anterior = productos_unicos[identificador]

                    categorias_previas = (
                        anterior.get("categorias_encontradas", "")
                    )

                    if nombre_categoria not in categorias_previas:

                        anterior["categorias_encontradas"] = (
                            categorias_previas
                            + " | "
                            + nombre_categoria
                        )

                total_resultados += 1

            print(
                f"Pagina {paginas_consultadas}: "
                f"{len(contenido)} resultados | "
                f"Nuevos: {nuevos_en_pagina} | "
                f"Acumulados: {len(productos_unicos)}"
            )

            time.sleep(ESPERA)

        except Exception as e:

            agregar_error(
                "Extraccion",
                nombre_categoria,
                respuesta.url if "respuesta" in locals() else URL_API,
                e
            )

            print("Error:", e)
            break

    resumen_categorias.append({

        "ID_categoria": categoria_id,

        "Nombre_categoria": nombre_categoria,

        "URL_categoria": categoria["url_categoria"],

        "Resultados_recuperados": total_resultados,

        "Paginas_consultadas": paginas_consultadas,

        "Productos_unicos_acumulados": len(productos_unicos)

    })

    print(
        "Total de resultados de la categoria:",
        total_resultados
    )

# Construir la Base de Productos

Este bloque es la Etapa 3: Aquí es donde la información cruda que estaba en el diccionario productos_unicos se convierte formalmente en la tabla (DataFrame) que luego será el Excel.

---
* Crea la tabla (DataFrame): Toma todos los diccionarios de productos que se guardaron en la Etapa 2 y los convierte en una tabla de Pandas (como una hoja de cálculo virtual).

* Limpia los datos complejos: Algunos campos (como las etiquetas, las categorías o los enlaces relacionados) venían como listas o diccionarios de Python, pero Excel no entiende eso. Esta parte usa **json.dumps** para convertir esas listas en texto plano (formato JSON), para que se puedan guardar sin errores en el Excel.

* Agrega el ID de registro: Inserta una columna nueva al principio llamada **ID_registro** con números consecutivos (1, 2, 3...) para que cada fila tenga un identificador único y ordenad

In [ ]:
print("\n" + "=" * 65)
print("ETAPA 3: CONSTRUCCION DE LA BASE")
print("=" * 65)

df = pd.DataFrame(
    list(productos_unicos.values())
)

if not df.empty:

    # Convertir listas y diccionarios para Excel.
    for columna in df.columns:

        df[columna] = df[columna].apply(

            lambda valor: json.dumps(
                valor,
                ensure_ascii=False
            )
            if isinstance(valor, (dict, list))
            else valor

        )

    df.insert(
        0,
        "ID_registro",
        range(1, len(df) + 1)
    )

# Traducción de Encabezados

**Traduce los encabezados:** Cambia los nombres técnicos que venían de la API (en inglés y en formato "camelCase", como ecommTitle o currentPrice) a nombres limpios y entendibles en español (Nombre_producto, Precio_actual). *Esto hace que la tabla final sea legible para cualquier persona.*

**Crea la tabla de categorías:** Convierte el resumen de categorías que se guardó en la Etapa 2 en un DataFrame (para la segunda hoja del Excel).

**Crea la tabla de errores:** Convierte la lista de errores en un DataFrame con columnas específicas (para la tercera hoja del Excel).


In [ ]:
traduccion_productos = {

    "ID_registro": "ID_registro",
    "ecommCatalogExtended": "Catalogo_ampliado",
    "tagging": "Etiquetas",
    "ecommP2Dropship": "Envio_proveedor",
    "ecommKeywords": "Palabras_clave",
    "ecommShippingDuration": "Tiempo_envio",
    "ecommTitle": "Nombre_producto",
    "score": "Puntuacion_busqueda",
    "ecommImageFullUrl": "URL_imagen_completa",
    "ecommTagAccumulate": "Etiquetas_acumuladas",
    "ecommTopProduct": "Producto_destacado",
    "ecommIsDonation": "Es_donacion",
    "ownBrand": "Es_marca_propia",
    "previousPrice": "Precio_anterior",
    "categories": "Categorias",
    "percentRewards": "Porcentaje_recompensa",
    "sku": "SKU",
    "stock": "Existencias_reportadas",
    "ecommP3Marketplace": "Es_marketplace",
    "ecommReviewProducts": "Productos_relacionados_resenas",
    "eb_categories": "Categorias_adicionales",
    "ecommBrand": "Marca",
    "ecommProductLinks": "Enlaces_producto",
    "ecommP1Fda": "Producto_Farmacias_Ahorro",
    "ecommIsStarProduct": "Es_producto_destacado",
    "currentPrice": "Precio_actual",
    "facetEcommMsi": "Facetas_MSI",
    "ecommMsi": "MSI",
    "categoryIds": "ID_categorias",
    "prescriptionRequired": "Requiere_receta",
    "variantGroupId": "ID_grupo_variantes",
    "ecommNumberOfReviews": "Numero_resenas",
    "ecommUrlKey": "Identificador_URL",
    "ecommId": "ID_producto",
    "ecommIsOptical": "Es_producto_optico",
    "orin": "ORIN",
    "ecommShortDescription": "Descripcion_corta",
    "URL_producto": "URL_producto",
    "URL_categoria": "URL_categoria",
    "categoria_origen": "Categoria_origen",
    "category_id_origen": "ID_categoria_origen",
    "categorias_encontradas": "Categorias_encontradas",
    "fecha_extraccion": "Fecha_extraccion",
    "pagina_origen": "Pagina_origen",
    "offers": "Ofertas",
    "ecommBenefitsOrUses": "Beneficios_o_usos",
    "variants": "Variantes",
    "ecommImageUrlGrouper": "URL_imagen_agrupador",
    "ecommTitleGrouper": "Nombre_agrupador",
    "ecommIdGrouper": "ID_agrupador",
    "urlKeyGrouper": "URL_agrupador",
    "ecommSimilarProductsSkus": "SKU_productos_similares"

}

df.rename(
    columns=traduccion_productos,
    inplace=True
)

df_categorias = pd.DataFrame(resumen_categorias)

df_errores = pd.DataFrame(
    errores,
    columns=[
        "Etapa_del_proceso",
        "Categoria",
        "URL_consultada",
        "Descripcion_del_error"
    ]
)

# Exportar a Excel

Etapa 4. El trabajo se guarda en el archivo Excel.

---

Crea el archivo Excel **"Farmacias_del_Ahorro_5500.xlsx"** usando la librería *openpyxl*.

Guarda las tres hojas que se construyeron en los pasos anteriores: Productos, Categorias y Errores.

Da formato profesional:

* Congela la primera fila (freeze_panes = "A2") para que *los encabezados siempre se vean al bajar.*

* Activa los filtros automáticos para que puedan ordenar y buscar.

* Ajusta el ancho de las columnas automáticamente para que el texto no se vea cortado.

* **Crea hipervínculos:** Busca la columna URL_producto y convierte cada enlace en un hipervínculo azul cliqueable. Esto es lo que permite que el usuario haga clic y verifique la página real de cada producto.

In [ ]:
print("\n" + "=" * 65)
print("ETAPA 4: EXPORTACION A EXCEL")
print("=" * 65)

with pd.ExcelWriter(
    NOMBRE_ARCHIVO,
    engine="openpyxl"
) as escritor:

    df.to_excel(
        escritor,
        sheet_name="Productos",
        index=False
    )

    df_categorias.to_excel(
        escritor,
        sheet_name="Categorias",
        index=False
    )

    df_errores.to_excel(
        escritor,
        sheet_name="Errores",
        index=False
    )

    for hoja in escritor.book.worksheets:

        hoja.freeze_panes = "A2"

        hoja.auto_filter.ref = hoja.dimensions

        for columna in hoja.columns:

            letra = columna[0].column_letter

            ancho = min(

                max(
                    len(str(celda.value or ""))
                    for celda in columna[:100]
                ) + 2,

                45

            )

            hoja.column_dimensions[letra].width = ancho

    # Crear hipervinculos en la hoja Productos.
    hoja = escritor.sheets["Productos"]

    posiciones = {
        celda.value: celda.column
        for celda in hoja[1]
    }

    if "URL_producto" in posiciones:

        columna_url = posiciones["URL_producto"]

        for fila in range(2, hoja.max_row + 1):

            celda = hoja.cell(
                row=fila,
                column=columna_url
            )

            if (
                celda.value
                and str(celda.value).startswith("https://")
            ):

                celda.hyperlink = celda.value
                celda.style = "Hyperlink"

# Resultados
* Imprime en pantalla los totales: Cuántas categorías se encontraron, cuántos productos únicos se recuperaron, si se llegó al límite de 5,500, cuántos errores hubo y el nombre del archivo generado.

* Avisa si faltó algo: Si no se llegó a los 5,500 productos, lanza un aviso de que se guardaron todos los que se pudieron recuperar.

* Muestra una vista previa: Si la tabla no está vacía, muestra en pantalla las primeras 10 filas con las columnas más importantes *(Nombre, Marca, Precio, URL)* usando el comando display. De este modo confirmamos visualmente que los datos tienen sentido.

In [ ]:
print("\n" + "=" * 65)
print("RESULTADOS DEL WEBSCRAPING")
print("=" * 65)

print("Categorias identificadas:", len(categorias))

print("Productos unicos recuperados:", len(df))

print("Limite solicitado:", MAX_PRODUCTOS)

print("Errores registrados:", len(df_errores))

print("Archivo generado:", NOMBRE_ARCHIVO)

if len(df) < MAX_PRODUCTOS:

    print(
        "\nAVISO: No se alcanzo el limite de 5,500. "
        "Se guardaron todos los productos recuperados."
    )

else:

    print("\nSe alcanzo el limite de 5,500 productos.")

if not df.empty:

    columnas_muestra = [
        columna for columna in [
            "ID_registro",
            "Nombre_producto",
            "SKU",
            "Marca",
            "Precio_actual",
            "Precio_anterior",
            "URL_producto"
        ]
        if columna in df.columns
    ]

    display(
        df[columnas_muestra].head(10)
    )

print("\nProceso terminado.")


ETAPA 1: DESCUBRIMIENTO DE CATEGORIAS
Codigo HTTP: 200
Paginas candidatas: 73
Categoria 1: Farmacias del Ahorro | Farmacia | Tienda en línea a todo México | ID: 7404
Categoria 2: Farmacias del Ahorro | Antitabaco - Farmacia | Tienda en línea a todo México | ID: 9315
Categoria 3: Farmacias del Ahorro | Controla los Síntomas - Antitabaco - Farmacia | Tienda en línea a todo México | ID: 11904
Categoria 4: Farmacias del Ahorro | Recupera tu Imagen  - Antitabaco - Farmacia | Tienda en línea a todo México | ID: 12834
Categoria 5: Farmacias del Ahorro | Recupera tu Salud - Antitabaco - Farmacia | Tienda en línea a todo México | ID: 9318
Categoria 6: Farmacias del Ahorro | Articulos de medición - Farmacia | Tienda en línea a todo México | ID: 7407
Categoria 7: Farmacias del Ahorro | Bienestar sexual - Farmacia | Tienda en línea a todo México | ID: 7839
Categoria 8: Farmacias del Ahorro | Accesorios sexualidad - Bienestar sexual - Farmacia | Tienda en línea a todo México | ID: 10365
Categoria 9

,ID_registro,Nombre_producto,SKU,Marca,Precio_actual,Precio_anterior,URL_producto
0,1,Esoxx One 20 Unidades 10 ml,8020030095076,ESOXX-ONE,465.500000,665.00,https://www.fahorro.com/esoxx-one-20-unidades-...
1,2,Riopan Gel Oral 20 sobres,7501092793045,RIOPAN,286.000000,286.00,https://www.fahorro.com/riopan-gel-oral-20-sob...
2,3,Omeprazol 20 mg Oral 30 cápsulas 2 Cajas Marca...,7502223709348,Marca del Ahorro,151.000000,151.00,https://www.fahorro.com/omeprazol-20-mg-oral-3...
3,4,Neuralin Solución Inyectable 2 pz,7501088505126,NEURALIN,147.050003,233.42,https://www.fahorro.com/neuralin-solucion-inye...
4,5,Alka Seltzer Alivio del Malestar Estomacal y D...,7501008443033,ALKA-SELTZER,48.000000,48.00,https://www.fahorro.com/alka-seltzer-antiacido...
5,6,Pinaverio Dimeticona 16 cápsulas Marca del Ahorro,7506472800788,Marca del Ahorro,148.000000,148.00,https://www.fahorro.com/pinaverio-dimeticona-1...
6,7,Enterogermina Suspensión de Bacillus Clausii c...,7501328980119,KRUPS DOLCE GUSTO,440.000000,440.00,https://www.fahorro.com/enterogermina-probioti...
7,8,"Dolo-Neurobión Forte DC Diclofenaco Sódico, Co...",7501298217635,DOLO NEUROBION,460.000000,460.00,https://www.fahorro.com/dolo-neurobion-forte-d...
8,9,Ibuprofeno 600 mg/10 cápsulas Marca del Ahorro,7502223705586,Marca del Ahorro,85.000000,85.00,https://www.fahorro.com/ibuprofeno-600-mg-10-c...
9,10,Actron 600 mg con 10 Caps,7501318652453,KRUPS DOLCE GUSTO,139.000000,139.00,https://www.fahorro.com/actron-600-mg-con-10-c...



Proceso terminado.



# Descarga del Excel

Como se trabajó en Google Colab, el archivo Excel se genera en la nube, no en la computadora, por lo que usamos:

**from google.colab import files:** Trae la herramienta de Colab para manejar archivos.

**files.download(...):** Toma el archivo Excel que acaba de crear el programa y lo descarga automáticamente en la computadora del usuario.




In [ ]:
from google.colab import files

files.download("Farmacias_del_Ahorro_5500.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>